# PDCP · N02 — Feasibility map and confound baseline
**Notebook 2 of 13.** Kaggle settings: Accelerator **None (CPU)** · Internet **On** · Run with **Save & Run All (Commit)**. Expected runtime 45–75 min.

**Input:** N01 output (attach via *Add Input → Notebook Output → your N01 notebook*). The notebook finds `pdcp_utils.py` automatically.

**Design decisions applied here (from N01 results, 13 Nov 2026)**
- Natural stress window **KSM 0.30–0.50** for all families (above 0.50 drawings degenerate into strips/clumps).
- Stress-difference levels for N03: **0, 0.03, 0.06, 0.10, 0.15, 0.20**.
- Realism limit `dense_max` raised 0.25 → **0.35** (proxy edits that make nodes less uniform are legitimate; true clumps measure ≥ 0.39). Module saved as **v0.1.1**.

**What it does**
1. **Base drawings:** for every family/size/graph and stress level in the window, a natural drawing with only stress targeted.
2. **Feasibility of proxy edits:** from each base drawing, try to move ONE proxy (NU, ELD or XR) by δ while holding stress and the other two proxies fixed. Feasible = converged + realistic. This is exactly the operation N03 needs for conflict pairs and counterfactual edits.
3. **Confound baseline:** correlation of stress with each proxy (a) in our natural base drawings and (b) in *Mooney-style* drawings (stress-only optimisation, 0.40–0.80, no realism constraints). Shows how strong the confound is before decorrelation.
4. Writes `design_config.json`: the per-family δ values N03 is allowed to use.

**Outputs** (`/kaggle/working/n02_output/`): `pdcp_utils.py` (v0.1.1), `base_drawings.parquet`, `feasibility.csv`, `feasibility_summary.csv`, `confound_baseline.csv`, `mooney_style.parquet`, `design_config.json`, `n02_report.json`, `fig_*.png`, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS (the only cell you ever need to edit) ====
INPUT_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n1-output/n01_output"   # N01 output folder
OUTPUT_DIR = "/kaggle/working/n02_output"
SEED = 2026
FAST = False                            # True = smoke test (~5 min)
N_JOBS = 4                              # Kaggle CPU = 4 cores

STRESS_LEVELS = [0.30, 0.35, 0.40, 0.45, 0.50]
GRAPHS_PER_SET = 3
FAMILY_SETS = [("ER", 25), ("ER", 50), ("ER", 100), ("BA", 50), ("BA", 100),
               ("SBM", 50), ("SBM", 100), ("WS", 50), ("TREE", 50), ("GRID", 49)]
DELTAS = {"nu":  [-0.10, -0.05, 0.05, 0.10, 0.15, 0.20],
          "eld": [-0.15, -0.10, -0.05, 0.05, 0.10, 0.15, 0.20],
          "xr":  [-0.04, -0.02, 0.02, 0.04, 0.06, 0.08]}
MOONEY_LEVELS = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
FEASIBLE_RATE = 0.80                    # a delta is usable for a family if >= 80% of its edits succeed

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib, re
os.makedirs(OUTPUT_DIR, exist_ok=True)
for p, pn in [("joblib", None), ("pyarrow", None)]:
    try: importlib.import_module(p)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pn or p], check=True)
import numpy as np, pandas as pd, networkx as nx, torch, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# --- locate N01's module (prefer a v0.1.x file inside an n01_output folder) ---
cands = sorted(glob.glob(f"{INPUT_ROOT}/**/pdcp_utils.py", recursive=True))
if not cands:
    raise FileNotFoundError("pdcp_utils.py not found under /kaggle/input — attach N01's output via Add Input.")
src_path = next((c for c in cands if "n01" in c.lower()), cands[0])
print("using", src_path)
src = open(src_path).read()

# --- apply agreed patch -> v0.1.1 ---
src = src.replace('__version__ = "0.1.0"', '__version__ = "0.1.1"')
src = src.replace('"dense_max": 0.25}', '"dense_max": 0.35}')
assert '"dense_max": 0.35}' in src and '"0.1.1"' in src, "patch failed: unexpected pdcp_utils.py contents"
UTILS = f"{OUTPUT_DIR}/pdcp_utils.py"
open(UTILS, "w").write(src)
sys.path.insert(0, OUTPUT_DIR)
import pdcp_utils as U; importlib.reload(U)
print("pdcp_utils", U.__version__, U.REALISM_LIMITS)
if FAST:
    FAMILY_SETS = FAMILY_SETS[:3] + [("GRID", 49)]; GRAPHS_PER_SET = 1; STRESS_LEVELS = [0.30, 0.40, 0.50]
    DELTAS = {k: v[::2] for k, v in DELTAS.items()}; MOONEY_LEVELS = MOONEY_LEVELS[::2]
T0 = time.time()

using /kaggle/input/datasets/resadussalehinrafsan/kbs-n1-output/n01_output/pdcp_utils.py
pdcp_utils 0.1.1 {'aspect_min': 0.7, 'nn_max_max': 2.6, 'nn_min_min': 0.2, 'dense_max': 0.35}


## 1. Base drawings (stress only, realistic)

In [3]:
def make_base(fam, n, gi, s):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import torch, pdcp_utils as U
    torch.set_num_threads(1)
    gseed = SEED + 100 * gi
    gc = U.GraphCache(U.make_graph(fam, n, gseed))
    for attempt in range(4):
        r = U.pdsg_optimise(gc, {"ksm": s}, seed=gseed + 7 * attempt + int(s * 1000))
        m = U.all_metrics(r.X, gc, with_realism=True)
        if r.converged and m["realistic"]:
            break
    return {"family": fam, "n": n, "graph": gi, "graph_seed": gseed, "stress_target": s,
            "converged": r.converged, "realistic": m["realistic"], "attempts": attempt + 1,
            "seconds": round(r.seconds, 2), **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "aspect", "nn_max", "nn_min", "dense"]},
            "coords": r.X.ravel().round(6).tolist()}

jobs = [(f, n, gi, s) for f, n in FAMILY_SETS for gi in range(GRAPHS_PER_SET) for s in STRESS_LEVELS]
print(len(jobs), "base drawings")
BASE = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(delayed(make_base)(*j) for j in jobs))
BASE["ok"] = BASE.converged & BASE.realistic
BASE.to_parquet(f"{OUTPUT_DIR}/base_drawings.parquet", index=False)
print(BASE.groupby(["family", "n"]).ok.mean().round(2).to_string())
print(f"{time.time()-T0:.0f}s")

150 base drawings


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   16.1s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  4.6min


family  n  
BA      50     1.00
        100    0.80
ER      25     1.00
        50     1.00
        100    0.87
GRID    49     1.00
SBM     50     1.00
        100    0.87
TREE    50     1.00
WS      50     1.00
512s


[Parallel(n_jobs=4)]: Done 150 out of 150 | elapsed:  8.5min finished


## 2. Feasibility of single-proxy edits (stress and other proxies fixed)

In [4]:
PROX = ["nu", "eld", "xr"]
def try_edit(row, proxy, delta):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import torch, numpy as np, pdcp_utils as U
    torch.set_num_threads(1)
    gc = U.GraphCache(U.make_graph(row["family"], row["n"], row["graph_seed"]))
    X0 = np.array(row["coords"]).reshape(-1, 2)
    base = {k: row[k] for k in ["ksm", "nu", "eld", "xr"]}
    tgt = dict(base); tgt[proxy] = base[proxy] + delta
    out = {"family": row["family"], "n": row["n"], "graph": row["graph"], "stress": row["stress_target"],
           "proxy": proxy, "delta": delta, "base_value": base[proxy], "target_value": tgt[proxy]}
    if tgt[proxy] < 0 or (proxy in ("nu", "xr") and tgt[proxy] > 1):
        return {**out, "converged": False, "realistic": False, "feasible": False, "reason": "target out of range", "seconds": 0.0}
    r = U.pdsg_optimise(gc, tgt, X0=X0, max_steps=1200, max_restarts=2, seed=int(1e4 * abs(delta)) + row["graph_seed"])
    m = U.all_metrics(r.X, gc, with_realism=True)
    reason = "ok" if (r.converged and m["realistic"]) else ("no convergence" if not r.converged else
             "unrealistic:" + ",".join(k for k, bad in [("aspect", m["aspect"] < U.REALISM_LIMITS["aspect_min"]),
                                                        ("outlier", m["nn_max"] > U.REALISM_LIMITS["nn_max_max"]),
                                                        ("overlap", m["nn_min"] < U.REALISM_LIMITS["nn_min_min"]),
                                                        ("clump", m["dense"] > U.REALISM_LIMITS["dense_max"])] if bad))
    return {**out, "converged": r.converged, "realistic": m["realistic"], "feasible": r.converged and m["realistic"],
            "reason": reason, "seconds": round(r.seconds, 2),
            **{"got_" + k: m[k] for k in ["ksm", "nu", "eld", "xr"]}, "dense": m["dense"], "nn_min": m["nn_min"]}

okb = BASE[BASE.ok].to_dict("records")
ejobs = [(row, p, d) for row in okb for p in PROX for d in DELTAS[p]]
print(len(ejobs), "edit attempts")
FE = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(delayed(try_edit)(*j) for j in ejobs))
FE.to_csv(f"{OUTPUT_DIR}/feasibility.csv", index=False)
FE["fam"] = FE.family + FE.n.astype(str)
SUM = FE.groupby(["fam", "proxy", "delta"]).agg(rate=("feasible", "mean"), n=("feasible", "size"),
                                               sec=("seconds", "median")).reset_index()
SUM.to_csv(f"{OUTPUT_DIR}/feasibility_summary.csv", index=False)
print(SUM.pivot_table(index="fam", columns=["proxy", "delta"], values="rate").round(2).to_string())
print(FE[~FE.feasible].reason.value_counts().head(10))
print(f"{time.time()-T0:.0f}s")

2717 edit attempts


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    3.4s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:   26.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done 280 tasks      | elapsed:  2.1min
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:  4.7min
[Parallel(n_jobs=4)]: Done 640 tasks      | elapsed:  8.5min
[Parallel(n_jobs=4)]: Done 874 tasks      | elapsed: 14.0min
[Parallel(n_jobs=4)]: Done 1144 tasks      | elapsed: 19.1min
[Parallel(n_jobs=4)]: Done 1450 tasks      | elapsed: 25.1min
[Parallel(n_jobs=4)]: Done 1792 tasks      | elapsed: 35.9min
[Parallel(n_jobs=4)]: Done 2170 tasks      | elapsed: 42.0min
[Parallel(n_jobs=4)]: Done 2584 tasks      | elapsed: 46.3min


proxy    eld                                        nu                                  xr                              
delta  -0.15 -0.10 -0.05  0.05  0.10  0.15  0.20 -0.10 -0.05  0.05  0.10  0.15  0.20 -0.04 -0.02  0.02  0.04  0.06  0.08
fam                                                                                                                     
BA100    1.0   1.0   1.0   1.0   1.0   1.0   1.0  0.00  1.00  1.00  0.83  0.58  0.00  1.00  1.00   1.0   1.0   1.0   1.0
BA50     1.0   1.0   1.0   1.0   1.0   1.0   1.0  0.00  1.00  1.00  0.87  0.60  0.00  0.73  1.00   1.0   1.0   1.0   1.0
ER100    1.0   1.0   1.0   1.0   1.0   1.0   1.0  0.00  1.00  1.00  0.69  0.31  0.00  0.85  1.00   1.0   1.0   1.0   1.0
ER25     1.0   1.0   1.0   1.0   1.0   1.0   1.0  0.93  1.00  0.93  0.93  0.67  0.33  0.93  1.00   1.0   1.0   1.0   1.0
ER50     1.0   1.0   1.0   1.0   1.0   1.0   1.0  0.07  1.00  1.00  0.87  0.27  0.00  0.80  1.00   1.0   1.0   1.0   1.0
GRID49   1.0   1.0   1.0   1.0  

[Parallel(n_jobs=4)]: Done 2717 out of 2717 | elapsed: 47.5min finished


## 3. Confound baseline: ours (natural, 0.30–0.50) vs Mooney-style (stress-only, 0.40–0.80)

In [5]:
def mooney_style(fam, n, gi, s):
    import sys; sys.path.insert(0, OUTPUT_DIR)
    import torch, pdcp_utils as U
    torch.set_num_threads(1)
    gseed = SEED + 100 * gi
    gc = U.GraphCache(U.make_graph(fam, n, gseed))
    # stress-only optimisation, realism and spacing switched off (as in the reference studies' generator)
    r = U.pdsg_optimise(gc, {"ksm": s}, seed=gseed + int(s * 1000), lam_overlap=0.0, lam_realism=0.0)
    m = U.all_metrics(r.X, gc, with_realism=True)
    return {"family": fam, "n": n, "graph": gi, "graph_seed": gseed, "stress_target": s, "converged": r.converged,
            **{k: m[k] for k in ["ksm", "nu", "eld", "xr", "aspect", "nn_max", "nn_min", "dense", "realistic"]},
            "coords": r.X.ravel().round(6).tolist()}

mjobs = [(f, n, gi, s) for f, n in FAMILY_SETS for gi in range(GRAPHS_PER_SET) for s in MOONEY_LEVELS]
MO = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(delayed(mooney_style)(*j) for j in mjobs))
MO.to_parquet(f"{OUTPUT_DIR}/mooney_style.parquet", index=False)

from scipy.stats import spearmanr
def corr_table(df, label):
    rows = []
    for fam, g in [("ALL", df)] + list(df.groupby(df.family + df.n.astype(str))):
        r = {"set": label, "family": fam, "n_drawings": len(g)}
        for p in PROX:
            r["rho_ksm_" + p] = spearmanr(g.ksm, g[p])[0] if g[p].nunique() > 1 else np.nan
        r["realistic_rate"] = g.realistic.mean() if "realistic" in g else np.nan
        rows.append(r)
    return pd.DataFrame(rows)
CONF = pd.concat([corr_table(BASE[BASE.ok], "ours_natural_0.30-0.50"),
                  corr_table(MO[MO.converged], "mooney_style_0.40-0.80")])
CONF.to_csv(f"{OUTPUT_DIR}/confound_baseline.csv", index=False)
print(CONF[CONF.family == "ALL"].round(3).to_string(index=False))
print("Mooney-style drawings passing realism:", round(MO.realistic.mean(), 3),
      "| by stress:", MO.groupby("stress_target").realistic.mean().round(2).to_dict())
print(f"{time.time()-T0:.0f}s")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    9.4s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   25.2s
[Parallel(n_jobs=4)]: Done 270 out of 270 | elapsed:   43.3s finished


                   set family  n_drawings  rho_ksm_nu  rho_ksm_eld  rho_ksm_xr  realistic_rate
ours_natural_0.30-0.50    ALL         143       0.415        0.570       0.866           1.000
mooney_style_0.40-0.80    ALL         270       0.954        0.863       0.198           0.011
Mooney-style drawings passing realism: 0.011 | by stress: {0.4: 0.1, 0.45: 0.0, 0.5: 0.0, 0.55: 0.0, 0.6: 0.0, 0.65: 0.0, 0.7: 0.0, 0.75: 0.0, 0.8: 0.0}
3405s


## 4. Design config for N03

In [6]:
FAMS = sorted(SUM.fam.unique())
cfg = {"utils_version": U.__version__, "stress_window": [min(STRESS_LEVELS), max(STRESS_LEVELS)],
       "dstress_levels": [0.0, 0.03, 0.06, 0.10, 0.15, 0.20], "same_threshold": 0.005,
       "feasible_rate_threshold": FEASIBLE_RATE, "families": {}}
for fam in FAMS:
    f = {}
    for p in PROX:
        s = SUM[(SUM.fam == fam) & (SUM.proxy == p)]
        good = s[s.rate >= FEASIBLE_RATE].delta.tolist()
        f[p] = {"usable_deltas": sorted(good),
                "max_up": max([d for d in good if d > 0], default=None),
                "max_down": min([d for d in good if d < 0], default=None)}
    b = BASE[(BASE.family + BASE.n.astype(str) == fam) & BASE.ok]
    f["base_ok_rate"] = float(BASE[BASE.family + BASE.n.astype(str) == fam].ok.mean())
    f["natural_proxy_ranges"] = {p: [float(b[p].quantile(.05)), float(b[p].quantile(.95))] for p in PROX}
    cfg["families"][fam] = f
# counterfactual magnitudes: largest common set of upward deltas usable in >= 70% of families, per proxy
cf = {}
for p in PROX:
    ups = sorted({d for fam in FAMS for d in cfg["families"][fam][p]["usable_deltas"] if d > 0})
    share = {d: np.mean([d in cfg["families"][fam][p]["usable_deltas"] for fam in FAMS]) for d in ups}
    cf[p] = [d for d in ups if share[d] >= 0.7][:3]
cfg["counterfactual_deltas"] = cf
cfg["conflict_margin"] = {"nu": 0.05, "eld": 0.05, "xr": 0.02}   # proxy must favour the higher-stress drawing by >= this
json.dump(cfg, open(f"{OUTPUT_DIR}/design_config.json", "w"), indent=2)
print(json.dumps({"counterfactual_deltas": cf, "stress_window": cfg["stress_window"]}, indent=1))
for fam in FAMS: print(fam, {p: cfg["families"][fam][p]["usable_deltas"] for p in PROX})

{
 "counterfactual_deltas": {
  "nu": [
   0.05,
   0.1
  ],
  "eld": [
   0.05,
   0.1,
   0.15
  ],
  "xr": [
   0.02,
   0.04,
   0.06
  ]
 },
 "stress_window": [
  0.3,
  0.5
 ]
}
BA100 {'nu': [-0.05, 0.05, 0.1], 'eld': [-0.15, -0.1, -0.05, 0.05, 0.1, 0.15, 0.2], 'xr': [-0.04, -0.02, 0.02, 0.04, 0.06, 0.08]}
BA50 {'nu': [-0.05, 0.05, 0.1], 'eld': [-0.15, -0.1, -0.05, 0.05, 0.1, 0.15, 0.2], 'xr': [-0.02, 0.02, 0.04, 0.06, 0.08]}
ER100 {'nu': [-0.05, 0.05], 'eld': [-0.15, -0.1, -0.05, 0.05, 0.1, 0.15, 0.2], 'xr': [-0.04, -0.02, 0.02, 0.04, 0.06, 0.08]}
ER25 {'nu': [-0.1, -0.05, 0.05, 0.1], 'eld': [-0.15, -0.1, -0.05, 0.05, 0.1, 0.15, 0.2], 'xr': [-0.04, -0.02, 0.02, 0.04, 0.06, 0.08]}
ER50 {'nu': [-0.05, 0.05, 0.1], 'eld': [-0.15, -0.1, -0.05, 0.05, 0.1, 0.15, 0.2], 'xr': [-0.04, -0.02, 0.02, 0.04, 0.06, 0.08]}
GRID49 {'nu': [-0.05, 0.05, 0.1], 'eld': [-0.15, -0.1, -0.05, 0.05, 0.1, 0.15, 0.2], 'xr': [-0.04, -0.02, 0.02, 0.04, 0.06, 0.08]}
SBM100 {'nu': [-0.05, 0.05, 0.1], 'eld': [-0

## 5. Figures

In [7]:
# F3a: feasibility heatmaps  (rows = family set, columns = proxy; cell = share of feasible edits)
fig, axes = plt.subplots(1, 3, figsize=(18, 0.45 * len(FAMS) + 2.2))
for ax, p in zip(axes, PROX):
    pv = SUM[SUM.proxy == p].pivot(index="fam", columns="delta", values="rate").reindex(FAMS)
    im = ax.imshow(pv.values, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels([f"{d:+.2f}" for d in pv.columns])
    ax.set_yticks(range(len(FAMS))); ax.set_yticklabels(FAMS)
    for i in range(pv.shape[0]):
        for j in range(pv.shape[1]):
            v = pv.values[i, j]
            if not np.isnan(v): ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{p.upper()} shift δ (stress + other proxies fixed)"); ax.set_xlabel("δ")
fig.colorbar(im, ax=axes, fraction=0.015, label="feasible share")
plt.savefig(f"{OUTPUT_DIR}/fig_F3a_feasibility.png", dpi=130, bbox_inches="tight"); plt.show()

# Confound: Spearman rho(KSM, proxy) — ours vs Mooney-style, per family
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
c = CONF[CONF.family != "ALL"]
for ax, p in zip(axes, PROX):
    piv = c.pivot(index="family", columns="set", values="rho_ksm_" + p)
    piv.plot.bar(ax=ax, color=["#d62728", "#1f77b4"], width=0.75, legend=(p == "nu"))
    ax.axhline(0, color="k", lw=.8); ax.set_title(f"Spearman ρ(KSM, {p.upper()})"); ax.set_ylim(-1, 1); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_confound_baseline.png", dpi=130); plt.show()

# Gallery of counterfactual edits: base + largest feasible up-shift per proxy (ER50, graph 0, stress 0.40)
def draw_from(row_coords, fam, n, gseed):
    gc = U.GraphCache(U.make_graph(fam, n, gseed)); return U.render_png(gc, np.array(row_coords).reshape(-1, 2), res=320, node_r=4)
ex = BASE[(BASE.family == "ER") & (BASE.n == 50) & (BASE.graph == 0) & BASE.ok]
if len(ex):
    row = ex.iloc[(ex.stress_target - 0.40).abs().argmin()].to_dict()
    panels = [("base  " + ", ".join(f"{k}={row[k]:.2f}" for k in ["ksm", "nu", "eld", "xr"]), draw_from(row["coords"], "ER", 50, row["graph_seed"]))]
    for p in PROX:
        d = (cf.get(p) or [None])[-1]
        if d is None: continue
        res = try_edit(row, p, d)
        if res["feasible"]:
            gc = U.GraphCache(U.make_graph("ER", 50, row["graph_seed"]))
            X0 = np.array(row["coords"]).reshape(-1, 2); tgt = {k: row[k] for k in ["ksm", "nu", "eld", "xr"]}; tgt[p] += d
            r = U.pdsg_optimise(gc, tgt, X0=X0, max_steps=1200, max_restarts=2, seed=int(1e4 * d) + row["graph_seed"])
            panels.append((f"{p.upper()} {d:+.2f}  " + ", ".join(f"{k}={r.metrics[k]:.2f}" for k in ["ksm", "nu", "eld", "xr"]),
                           U.render_png(gc, r.X, res=320, node_r=4)))
    fig, ax = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.6))
    for a, (t, im) in zip(np.atleast_1d(ax), panels): a.imshow(im); a.set_title(t, fontsize=8); a.axis("off")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_counterfactual_examples.png", dpi=120); plt.show()

# Mooney-style vs ours at matched stress (same graph)
mo = MO[(MO.family == "ER") & (MO.n == 50) & (MO.graph == 0)]
if len(mo):
    lv = [l for l in [0.40, 0.50, 0.60, 0.70, 0.80] if l in set(mo.stress_target)]
    fig, ax = plt.subplots(1, len(lv), figsize=(3.4 * len(lv), 3.8))
    for a, l in zip(np.atleast_1d(ax), lv):
        r = mo[mo.stress_target == l].iloc[0]
        a.imshow(draw_from(r.coords, "ER", 50, r.graph_seed)); a.axis("off")
        a.set_title(f"Mooney-style KSM {r.ksm:.2f}\nNU {r.nu:.2f} {'ok' if r.realistic else 'DEGENERATE'}", fontsize=9,
                    color="black" if r.realistic else "red")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_mooney_style_examples.png", dpi=120); plt.show()

ValueError: 2026 cannot be used to generate a random.Random instance

## 6. Report and manifest

In [ ]:
REPORT = {"utils_version": U.__version__, "runtime_min": round((time.time() - T0) / 60, 1), "fast": FAST,
          "base_ok_rate": float(BASE.ok.mean()), "edit_attempts": int(len(FE)), "edit_feasible_rate": float(FE.feasible.mean()),
          "failure_reasons": FE[~FE.feasible].reason.value_counts().to_dict(),
          "confound_all": CONF[CONF.family == "ALL"].drop(columns="family").to_dict("records"),
          "mooney_style_realistic_rate": float(MO.realistic.mean()),
          "mooney_style_realistic_by_stress": MO.groupby("stress_target").realistic.mean().round(3).to_dict(),
          "counterfactual_deltas": cfg["counterfactual_deltas"]}
json.dump(REPORT, open(f"{OUTPUT_DIR}/n02_report.json", "w"), indent=2, default=str)
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
desc = {"pdcp_utils.py": "shared module v0.1.1 (dense_max 0.35) - import this from now on",
        "base_drawings.parquet": "natural base drawings (coords + exact metrics + realism)",
        "feasibility.csv": "every single-proxy edit attempt", "feasibility_summary.csv": "feasible share per family/proxy/delta",
        "confound_baseline.csv": "Spearman rho(KSM, proxy): ours vs Mooney-style",
        "mooney_style.parquet": "stress-only drawings 0.40-0.80 (for E1b)", "design_config.json": "per-family usable deltas -> N03",
        "n02_report.json": "summary", "fig_F3a_feasibility.png": "feasibility heatmaps (paper Fig 3a)",
        "fig_confound_baseline.png": "confound bars", "fig_counterfactual_examples.png": "base vs single-proxy edits",
        "fig_mooney_style_examples.png": "Mooney-style drawings by stress"}
lines = [f"N02 output · utils {U.__version__} · runtime {REPORT['runtime_min']} min · base ok {REPORT['base_ok_rate']:.2f} "
         f"· edits feasible {REPORT['edit_feasible_rate']:.2f}", ""]
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(p): lines.append(f"{f:34s} {os.path.getsize(p)/1024:9.1f} KB   {desc.get(f, '')}")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))